In [8]:
import os, glob, math
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ======================
# === Config / layout ==
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

FUSION_RADARS = ["rpi4", "rpi1", "rpi3"]
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}  # yaw=0 for now
radar_name_map = {
    "rpi4": "Radar 1",
    "rpi1": "Radar 2",
    "rpi3": "Radar 3",
}

# ti_mmwave column indices (0-based)
CSV_X_COL       = 3
CSV_Y_COL       = 4
CSV_RANGE_COL   = 6
CSV_DOPPLER_COL = 8   # <--- here
CSV_ANGLE_COL   = 9
CSV_INTEN_COL   = 10
CSV_SNR_COL     = 11
CSV_NOISE_COL   = 12
CSV_TIME_COL    = -1

# DBSCAN + binning
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.20
DBSCAN_MINPTS    = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0  # default, but can be toggled

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
})

# ======================
# === Loading helpers ==
# ======================

def _has_header(path: str) -> bool:
    try:
        with open(path, "r", errors="ignore") as f:
            toks = f.readline().strip().split(",")
        for t in toks:
            try:
                float(t)
            except ValueError:
                return True
        return False
    except:
        return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:
        return pd.read_csv(path, header=header, sep=None, engine="python")
    except:
        return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec, int):
        idx = spec if spec >= 0 else df.shape[1] + spec
        return df.iloc[:, idx]
    if isinstance(spec, str):
        if spec in df.columns:
            return df[spec]
        low = {c.lower(): c for c in df.columns}
        if spec.lower() in low:
            return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str) -> List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact):
        return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact, "**", "*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root, "**", "*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f)
                and os.path.basename(f) == suffix]
        return sorted(hits)
    return []

def rot2d(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float,
                             tx: float, ty: float) -> pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x", "y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"] = xy[:, 0] + tx
    out["yw"] = xy[:, 1] + ty
    return out

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)

    out = pd.DataFrame({
        "x":     pd.to_numeric(_get(df, CSV_X_COL),       errors="coerce"),
        "y":     pd.to_numeric(_get(df, CSV_Y_COL),       errors="coerce"),
        "range": pd.to_numeric(_get(df, CSV_RANGE_COL),   errors="coerce"),
        "angle": pd.to_numeric(_get(df, CSV_ANGLE_COL),   errors="coerce"),
        "inten": pd.to_numeric(_get(df, CSV_INTEN_COL),   errors="coerce"),
        "snr":   pd.to_numeric(_get(df, CSV_SNR_COL),     errors="coerce"),
        "noise": pd.to_numeric(_get(df, CSV_NOISE_COL),   errors="coerce"),
    })
    try:
        out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL),
                                       errors="coerce")
    except Exception:
        out["doppler"] = np.nan
    try:
        out["timestamp"] = _get(df, CSV_TIME_COL)
    except Exception:
        out["timestamp"] = np.nan

    return out.dropna(subset=["x", "y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame()
    parts = []
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float),
                                    REMOVE_DOPPLER_EQ)]
            df["radar"] = radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts = []
    for r, (tx, ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty:
            continue
        yaw = float(sides.get(r, {}).get("angle", 0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x", "y", "range", "angle", "inten", "snr",
               "noise", "doppler", "timestamp", "radar", "xw", "yw"]]

# ======================
# === DBSCAN helpers ===
# ======================

def to_seconds(s: pd.Series) -> pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

def bin_time(df: pd.DataFrame, bin_s: float) -> pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"] / bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame) -> pd.DataFrame:
    if df_bin.empty:
        return pd.DataFrame()
    X = df_bin[["xw", "yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"] = labels
    return out

def compute_centroids(df_bin: pd.DataFrame):
    cents = {}
    for c, d in df_bin.groupby("cluster"):
        if c == -1:
            continue
        cents[c] = (float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# ===================================================
# === Raw vs clustered plot with doppler switch   ===
# ===================================================

def load_transform_suffix_all_radars_with_doppler(
    sfx: str,
    doppler_eq8: bool
) -> pd.DataFrame:
    global REMOVE_DOPPLER_EQ
    old_val = REMOVE_DOPPLER_EQ
    REMOVE_DOPPLER_EQ = 8.0 if doppler_eq8 else None
    try:
        df = load_transform_suffix_all_radars(sfx)
    finally:
        REMOVE_DOPPLER_EQ = old_val
    return df
def plot_raw_vs_clustered_suffix(
    suffix: str,
    doppler_eq8: bool = True,
    out_dir: str = "DBSCAN_plots",
    show: bool = False,
):
    os.makedirs(out_dir, exist_ok=True)

    df = load_transform_suffix_all_radars_with_doppler(suffix, doppler_eq8)
    if df.empty:
        print(f"[plot_raw_vs_clustered] suffix={suffix}: no data found.")
        return

    plt.figure(figsize=(8, 8))

    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    # ==== RAW POINTS ====
    for r in FUSION_RADARS:
        d_r = df[df["radar"] == r]
        if d_r.empty:
            continue
        plt.scatter(
            d_r["xw"], d_r["yw"],
            s=40,
            alpha=1.0,
            color=radar_colors[r],
            edgecolor="black",
            linewidths=0.4,
            label=f"{radar_name_map.get(r, r)} raw",
        )

    # ==== DBSCAN CENTROIDS ====
    df_binned = bin_time(df, BIN_SECONDS)
    cx, cy = [], []
    for _, d_bin in df_binned.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        for (x, y) in cents.values():
            cx.append(x)
            cy.append(y)

    if cx:
        plt.scatter(
            cx, cy,
            s=50,
            color="black",
            marker="X",
            edgecolor="white",
            linewidths=0.5,
            label="DBSCAN centroids",
        )

    # ==== RADAR ANCHORS ====
    for r in FUSION_RADARS:
        ax, ay = radar_positions[r]
        plt.scatter(
            ax, ay,
            s=250,
            marker="^",
            color=radar_colors[r],
            edgecolor="black",
            linewidths=1.3,
            label=f"{radar_name_map.get(r, r)} anchor",
        )

    plt.axis("equal")
    plt.xlabel("x (m)", fontsize=34)
    plt.ylabel("y (m)", fontsize=34)
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(loc="best", fontsize=16)

    out_path = os.path.join(out_dir, f"{suffix}_raw_vs_clustered.png")
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=28)
    plt.tick_params(axis='both', which='minor', labelsize=26)

    plt.savefig(out_path, dpi=500, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close()

    print(f"[plot_raw_vs_clustered] saved {out_path}")

# Examples
plot_raw_vs_clustered_suffix("rr5", doppler_eq8=False) 
plot_raw_vs_clustered_suffix("c6", doppler_eq8=False)
plot_raw_vs_clustered_suffix("cc2", doppler_eq8=False)
plot_raw_vs_clustered_suffix("r5", doppler_eq8=False)# doppler==8 removed
#plot_raw_vs_clustered_suffix("c6", doppler_eq8=False)  # keep all doppler


[plot_raw_vs_clustered] saved DBSCAN_plots/rr5_raw_vs_clustered.png
[plot_raw_vs_clustered] saved DBSCAN_plots/c6_raw_vs_clustered.png
[plot_raw_vs_clustered] saved DBSCAN_plots/cc2_raw_vs_clustered.png
[plot_raw_vs_clustered] saved DBSCAN_plots/r5_raw_vs_clustered.png


In [9]:
import os, glob, math
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ======================
# === Config / layout ==
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

FUSION_RADARS = ["rpi4", "rpi1", "rpi3"]
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}  # yaw=0 for now

# ti_mmwave column indices (0-based)
CSV_X_COL       = 3
CSV_Y_COL       = 4
CSV_RANGE_COL   = 6
CSV_DOPPLER_COL = 8   # <--- here
CSV_ANGLE_COL   = 9
CSV_INTEN_COL   = 10
CSV_SNR_COL     = 11
CSV_NOISE_COL   = 12
CSV_TIME_COL    = -1

# DBSCAN + binning
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.20
DBSCAN_MINPTS    = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0  # default, but can be toggled

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
})

# ======================
# === Loading helpers ==
# ======================

def _has_header(path: str) -> bool:
    try:
        with open(path, "r", errors="ignore") as f:
            toks = f.readline().strip().split(",")
        for t in toks:
            try:
                float(t)
            except ValueError:
                return True
        return False
    except:
        return False

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).

    Now also stores per-radar world coordinates (r_xw, r_yw) for plotting.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {
                "suffix": sfx,
                "tbin": int(tbin),
                "cluster": int(cid),
                "x_avg": float(points[:,0].mean()),
                "y_avg": float(points[:,1].mean()),
                "t_ref": float(np.mean(times)),
            }

            # per-radar features + world coordinates
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"] = float(srow[k])
                trip[f"{r}_xw"] = float(srow["xw"])
                trip[f"{r}_yw"] = float(srow["yw"])

            rows.append(trip)
    return pd.DataFrame(rows)

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:
        return pd.read_csv(path, header=header, sep=None, engine="python")
    except:
        return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec, int):
        idx = spec if spec >= 0 else df.shape[1] + spec
        return df.iloc[:, idx]
    if isinstance(spec, str):
        if spec in df.columns:
            return df[spec]
        low = {c.lower(): c for c in df.columns}
        if spec.lower() in low:
            return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str) -> List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact):
        return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact, "**", "*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root, "**", "*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f)
                and os.path.basename(f) == suffix]
        return sorted(hits)
    return []

def rot2d(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float,
                             tx: float, ty: float) -> pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x", "y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"] = xy[:, 0] + tx
    out["yw"] = xy[:, 1] + ty
    return out

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)

    out = pd.DataFrame({
        "x":     pd.to_numeric(_get(df, CSV_X_COL),       errors="coerce"),
        "y":     pd.to_numeric(_get(df, CSV_Y_COL),       errors="coerce"),
        "range": pd.to_numeric(_get(df, CSV_RANGE_COL),   errors="coerce"),
        "angle": pd.to_numeric(_get(df, CSV_ANGLE_COL),   errors="coerce"),
        "inten": pd.to_numeric(_get(df, CSV_INTEN_COL),   errors="coerce"),
        "snr":   pd.to_numeric(_get(df, CSV_SNR_COL),     errors="coerce"),
        "noise": pd.to_numeric(_get(df, CSV_NOISE_COL),   errors="coerce"),
    })
    try:
        out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL),
                                       errors="coerce")
    except Exception:
        out["doppler"] = np.nan
    try:
        out["timestamp"] = _get(df, CSV_TIME_COL)
    except Exception:
        out["timestamp"] = np.nan

    return out.dropna(subset=["x", "y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame()
    parts = []
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float),
                                    REMOVE_DOPPLER_EQ)]
            df["radar"] = radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts = []
    for r, (tx, ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty:
            continue
        yaw = float(sides.get(r, {}).get("angle", 0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x", "y", "range", "angle", "inten", "snr",
               "noise", "doppler", "timestamp", "radar", "xw", "yw"]]

# ======================
# === DBSCAN helpers ===
# ======================

def to_seconds(s: pd.Series) -> pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

def bin_time(df: pd.DataFrame, bin_s: float) -> pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"] / bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame) -> pd.DataFrame:
    if df_bin.empty:
        return pd.DataFrame()
    X = df_bin[["xw", "yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"] = labels
    return out

def compute_centroids(df_bin: pd.DataFrame):
    cents = {}
    for c, d in df_bin.groupby("cluster"):
        if c == -1:
            continue
        cents[c] = (float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# ===================================================
# === Raw vs clustered plot with doppler switch   ===
# ===================================================

def load_transform_suffix_all_radars_with_doppler(
    sfx: str,
    doppler_eq8: bool
) -> pd.DataFrame:
    global REMOVE_DOPPLER_EQ
    old_val = REMOVE_DOPPLER_EQ
    REMOVE_DOPPLER_EQ = 8.0 if doppler_eq8 else None
    try:
        df = load_transform_suffix_all_radars(sfx)
    finally:
        REMOVE_DOPPLER_EQ = old_val
    return df

def plot_triplet_points_with_dbscan_centroids(
    suffix: str,
    out_dir: str = "DBSCAN_plots",
    show: bool = False,
):
    """
    For a given suffix:
      - Runs the usual CSM + DBSCAN + min-dmid selection.
      - Plots ONLY the three radar points that form each selected triplet.
      - Overlays the corresponding DBSCAN cluster centroids.

    So you see:
      * points used in triplets (colored by radar)
      * cluster centroids (black 'X')
      * radar anchors (triangles)
    """
    os.makedirs(out_dir, exist_ok=True)

    trips = select_triplets_for_suffix(suffix)
    if trips.empty:
        print(f"[triplet plot] suffix={suffix}: no selected triplets.")
        return

    df_all = load_transform_suffix_all_radars(suffix)
    if df_all.empty:
        print(f"[triplet plot] suffix={suffix}: no detections.")
        return

    df_all_b = bin_time(df_all, BIN_SECONDS)

    # --- DBSCAN centroids per (tbin, cluster) ---
    cent_rows = []
    for tbin, d_bin in df_all_b.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        for cid, (cx, cy) in cents.items():
            cent_rows.append({
                "tbin": int(tbin),
                "cluster": int(cid),
                "cx": cx,
                "cy": cy,
            })
    if not cent_rows:
        print(f"[triplet plot] suffix={suffix}: no DBSCAN centroids.")
        return
    df_cent = pd.DataFrame(cent_rows)

    # only centroids whose (tbin,cluster) produced a triplet
    df_trip_keys = trips[["tbin", "cluster"]].drop_duplicates()
    df_cent_sel = df_cent.merge(df_trip_keys, on=["tbin", "cluster"], how="inner")

    plt.figure(figsize=(8, 8))

    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    # 1) Triplet points per radar
    for r in FUSION_RADARS:
        x_col = f"{r}_xw"
        y_col = f"{r}_yw"
        if x_col not in trips.columns or y_col not in trips.columns:
            continue
        plt.scatter(
            trips[x_col],
            trips[y_col],
            s=40,
            alpha=1.0,
            color=radar_colors.get(r, "C0"),
            edgecolor="black",
            linewidths=0.4,
            label=f"{radar_name_map.get(r, r)} triplet points",
        )

    # 2) DBSCAN centroids
    plt.scatter(
        df_cent_sel["cx"],
        df_cent_sel["cy"],
        s=50,
        color="black",
        marker="X",
        linewidths=0.5,
        label="DBSCAN centroids",
    )

    # 3) Radar anchors
    for r in FUSION_RADARS:
        if r not in radar_positions:
            continue
        ax, ay = radar_positions[r]
        plt.scatter(
            ax, ay,
            s=250,
            marker="^",
            color=radar_colors.get(r, "C0"),
            edgecolor="black",
            linewidths=1.3,
            label=f"{radar_name_map.get(r, r)} anchor",
        )

    plt.axis("equal")
    plt.xlabel("x (m)", fontsize=34)
    plt.ylabel("y (m)", fontsize=34)
    plt.grid(True, linestyle="--", alpha=0.35)
   # plt.title(f"{suffix} — triplet points + DBSCAN centroids",
    #          fontsize=16, weight="bold")
    plt.legend(loc="best", fontsize=16)

    out_path = os.path.join(out_dir, f"{suffix}_triplets_vs_centroids.png")
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=28)
    plt.tick_params(axis='both', which='minor', labelsize=26)
    plt.savefig(out_path, dpi=500, bbox_inches="tight")
    
    if show:
        plt.show()
    else:
        plt.close()
    print(f"[triplet plot] saved {out_path}")


# Examples
#plot_triplet_points_with_dbscan_centroids("c6")
#plot_triplet_points_with_dbscan_centroids("rr5")
plot_triplet_points_with_dbscan_centroids("cc2")
#plot_triplet_points_with_dbscan_centroids("r5")

#

NameError: name 'build_candidates_for_cluster' is not defined

In [11]:
import os, glob, math
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ======================
# === Config / layout ==
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

FUSION_RADARS = ["rpi4", "rpi1", "rpi3"]
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
}
radar_name_map = {
    "rpi4": "Radar 1",
    "rpi1": "Radar 2",
    "rpi3": "Radar 3",
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}  # yaw=0 for now

# ti_mmwave column indices (0-based)
CSV_X_COL       = 3
CSV_Y_COL       = 4
CSV_RANGE_COL   = 6
CSV_DOPPLER_COL = 8   # <--- here
CSV_ANGLE_COL   = 9
CSV_INTEN_COL   = 10
CSV_SNR_COL     = 11
CSV_NOISE_COL   = 12
CSV_TIME_COL    = -1

# DBSCAN + binning
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.20
DBSCAN_MINPTS    = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0  # default, but can be toggled

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
})

# ======================
# === Loading helpers ==
# ======================

def _has_header(path: str) -> bool:
    try:
        with open(path, "r", errors="ignore") as f:
            toks = f.readline().strip().split(",")
        for t in toks:
            try:
                float(t)
            except ValueError:
                return True
        return False
    except:
        return False

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).

    Now also stores per-radar world coordinates (r_xw, r_yw) for plotting.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {
                "suffix": sfx,
                "tbin": int(tbin),
                "cluster": int(cid),
                "x_avg": float(points[:,0].mean()),
                "y_avg": float(points[:,1].mean()),
                "t_ref": float(np.mean(times)),
            }

            # per-radar features + world coordinates
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"] = float(srow[k])
                trip[f"{r}_xw"] = float(srow["xw"])
                trip[f"{r}_yw"] = float(srow["yw"])

            rows.append(trip)
    return pd.DataFrame(rows)

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:
        return pd.read_csv(path, header=header, sep=None, engine="python")
    except:
        return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec, int):
        idx = spec if spec >= 0 else df.shape[1] + spec
        return df.iloc[:, idx]
    if isinstance(spec, str):
        if spec in df.columns:
            return df[spec]
        low = {c.lower(): c for c in df.columns}
        if spec.lower() in low:
            return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str) -> List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact):
        return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact, "**", "*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root, "**", "*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f)
                and os.path.basename(f) == suffix]
        return sorted(hits)
    return []

def rot2d(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float,
                             tx: float, ty: float) -> pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x", "y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"] = xy[:, 0] + tx
    out["yw"] = xy[:, 1] + ty
    return out

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)

    out = pd.DataFrame({
        "x":     pd.to_numeric(_get(df, CSV_X_COL),       errors="coerce"),
        "y":     pd.to_numeric(_get(df, CSV_Y_COL),       errors="coerce"),
        "range": pd.to_numeric(_get(df, CSV_RANGE_COL),   errors="coerce"),
        "angle": pd.to_numeric(_get(df, CSV_ANGLE_COL),   errors="coerce"),
        "inten": pd.to_numeric(_get(df, CSV_INTEN_COL),   errors="coerce"),
        "snr":   pd.to_numeric(_get(df, CSV_SNR_COL),     errors="coerce"),
        "noise": pd.to_numeric(_get(df, CSV_NOISE_COL),   errors="coerce"),
    })
    try:
        out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL),
                                       errors="coerce")
    except Exception:
        out["doppler"] = np.nan
    try:
        out["timestamp"] = _get(df, CSV_TIME_COL)
    except Exception:
        out["timestamp"] = np.nan

    return out.dropna(subset=["x", "y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame()
    parts = []
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float),
                                    REMOVE_DOPPLER_EQ)]
            df["radar"] = radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts = []
    for r, (tx, ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty:
            continue
        yaw = float(sides.get(r, {}).get("angle", 0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x", "y", "range", "angle", "inten", "snr",
               "noise", "doppler", "timestamp", "radar", "xw", "yw"]]

# ======================
# === DBSCAN helpers ===
# ======================

def to_seconds(s: pd.Series) -> pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

def bin_time(df: pd.DataFrame, bin_s: float) -> pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"] / bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame) -> pd.DataFrame:
    if df_bin.empty:
        return pd.DataFrame()
    X = df_bin[["xw", "yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"] = labels
    return out

def compute_centroids(df_bin: pd.DataFrame):
    cents = {}
    for c, d in df_bin.groupby("cluster"):
        if c == -1:
            continue
        cents[c] = (float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# ===================================================
# === Raw vs clustered plot with doppler switch   ===
# ===================================================

def load_transform_suffix_all_radars_with_doppler(
    sfx: str,
    doppler_eq8: bool
) -> pd.DataFrame:
    global REMOVE_DOPPLER_EQ
    old_val = REMOVE_DOPPLER_EQ
    REMOVE_DOPPLER_EQ = 8.0 if doppler_eq8 else None
    try:
        df = load_transform_suffix_all_radars(sfx)
    finally:
        REMOVE_DOPPLER_EQ = old_val
    return df

def plot_triplet_points_with_dbscan_centroids(
    suffix: str,
    out_dir: str = "DBSCAN_plots",
    show: bool = False,
):
    """
    For a given suffix:
      - Runs the usual CSM + DBSCAN + min-dmid selection.
      - Plots ONLY the three radar points that form each selected triplet.
      - Overlays the corresponding DBSCAN cluster centroids.

    So you see:
      * points used in triplets (colored by radar)
      * cluster centroids (black 'x')
      * radar anchors (triangles)
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- Selected triplets (now includes per-radar xw,yw) ---
    trips = select_triplets_for_suffix(suffix)
    if trips.empty:
        print(f"[triplet plot] suffix={suffix}: no selected triplets.")
        return

    # --- Full detections for this suffix (for DBSCAN & centroids) ---
    df_all = load_transform_suffix_all_radars(suffix)
    if df_all.empty:
        print(f"[triplet plot] suffix={suffix}: no detections.")
        return

    df_all_b = bin_time(df_all, BIN_SECONDS)

    # --- Compute DBSCAN centroids per (tbin, cluster) ---
    cent_rows = []
    for tbin, d_bin in df_all_b.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        for cid, (cx, cy) in cents.items():
            cent_rows.append({
                "tbin": int(tbin),
                "cluster": int(cid),
                "cx": cx,
                "cy": cy,
            })
    if not cent_rows:
        print(f"[triplet plot] suffix={suffix}: no DBSCAN centroids.")
        return
    df_cent = pd.DataFrame(cent_rows)

    # keep only centroids for clusters that actually produced a triplet
    df_trip_keys = trips[["tbin","cluster"]].drop_duplicates()
    df_cent_sel = df_cent.merge(df_trip_keys, on=["tbin","cluster"], how="inner")

    # --- Plot ---
    plt.figure(figsize=(8, 8))

    # Radar colors
    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    # 1) Triplet points per radar
    for r in FUSION_RADARS:
        x_col = f"{r}_xw"
        y_col = f"{r}_yw"
        if x_col not in trips.columns or y_col not in trips.columns:
            continue
        plt.scatter(
            trips[x_col],
            trips[y_col],
            s=40,
            alpha=1.0,
            color=radar_colors.get(r, "C0"),
            edgecolor="black",
            linewidths=0.4,
            label=f"{radar_name_map[r]} triplet points",
        )

    # 2) DBSCAN centroids (for those triplet clusters only)
    plt.scatter(
        df_cent_sel["cx"],
        df_cent_sel["cy"],
        s=50,
        color="black",
        marker="X",
        linewidths=0.5,
        label="DBSCAN centroids",
    )

    # 3) Radar anchors
    for r in FUSION_RADARS:
        if r not in radar_positions:
            continue
        ax, ay = radar_positions[r]
        plt.scatter(
            ax, ay,
            s=250,
            marker="^",
            color=radar_colors.get(r, "C0"),
            edgecolor="black",
            linewidths=1.3,
            label=f"{radar_name_map[r]} anchor",
        )


    # axes / cosmetics
    plt.axis("equal")
    plt.xlabel("x (m)", fontsize=34)
    plt.ylabel("y (m)", fontsize=34)
    plt.grid(True, linestyle="--", alpha=0.35)
  #  plt.title(f"{suffix} — triplet points + DBSCAN centroids",
   #           fontsize=16, weight="bold")
    plt.legend(loc="best", fontsize=16)

    out_path = os.path.join(out_dir, f"{suffix}_triplets_vs_centroids.png")
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=28)
    plt.tick_params(axis='both', which='minor', labelsize=26)
    plt.savefig(out_path, dpi=500, bbox_inches="tight")   
    if show:
        plt.show()
    else:
        plt.close()

    print(f"[triplet plot] saved {out_path}")

# ======================================================
# === Triplet candidate builder (REQUIRED FUNCTIONS) ===
# ======================================================

K_NEAREST_PER_RADAR = 3     # or whatever you used earlier
TIME_TOLERANCE_S = 0.30
SPREAD_MAX_M = 0.25

def nearest_k_to_centroid(df_r, centroid, k):
    """Return K closest detections of radar r to cluster centroid."""
    dx = df_r["xw"].to_numpy() - centroid[0]
    dy = df_r["yw"].to_numpy() - centroid[1]
    d2 = dx*dx + dy*dy
    idx = np.argsort(d2)[:k]
    return df_r.iloc[idx]

def build_candidates_for_cluster(d_cluster, centroid, radars, K):
    """
    Build candidate triplets as cartesian product of K-nearest per radar.
    Apply spread gate + time gate.
    """
    # --- K nearest per radar ---
    per_radar = {}
    for r in radars:
        dr = d_cluster[d_cluster["radar"] == r]
        if dr.empty:
            return []
        per_radar[r] = nearest_k_to_centroid(dr, centroid, K).reset_index(drop=True)

    # --- Cartesian product of indices ---
    cands = []
    for i in range(len(per_radar[radars[0]])):
        for j in range(len(per_radar[radars[1]])):
            for k in range(len(per_radar[radars[2]])):

                p1 = per_radar[radars[0]].iloc[i]
                p2 = per_radar[radars[1]].iloc[j]
                p3 = per_radar[radars[2]].iloc[k]

                pts = np.array([
                    [p1["xw"], p1["yw"]],
                    [p2["xw"], p2["yw"]],
                    [p3["xw"], p3["yw"]],
                ])

                # --- spread gate ---
                d12 = np.linalg.norm(pts[0] - pts[1])
                d23 = np.linalg.norm(pts[1] - pts[2])
                d31 = np.linalg.norm(pts[2] - pts[0])
                spread = max(d12, d23, d31)
                if spread > SPREAD_MAX_M:
                    continue

                # --- time gate ---
                t1 = float(p1["t"])
                t2 = float(p2["t"])
                t3 = float(p3["t"])
                if max(t1, t2, t3) - min(t1, t2, t3) > TIME_TOLERANCE_S:
                    continue

                cands.append({
                    "points": pts,
                    "times": [t1, t2, t3],
                    "rows": [i, j, k],   # needed later
                })

    return cands

def triplet_features_for_scoring(cand, centroid):
    """
    Compute features = [s12, s23, s31, spread, dmid]
    """
    P = np.asarray(cand["points"])
    d12 = np.linalg.norm(P[0] - P[1])
    d23 = np.linalg.norm(P[1] - P[2])
    d31 = np.linalg.norm(P[2] - P[0])
    spread = max(d12, d23, d31)

    # midpoint
    mid = np.mean(P, axis=0)
    dmid = np.linalg.norm(mid - centroid)

    return np.array([d12, d23, d31, spread, dmid])

def plot_all_triplets_and_selected_for_suffix(
    suffix: str,
    out_dir: str = "DBSCAN_triplets_debug",
    max_plots: int = None,
    show: bool = False,
):
    """
    For a given suffix:
      - Runs DBSCAN + candidate triplet generation (build_candidates_for_cluster).
      - For each (tbin, cluster):
          * Plots ALL candidate triplets (points + triangle edges).
          * Highlights the final selected triplet (min-dmid).
          * Marks the cluster centroid.
          * Shows radar anchors.

    One PNG per (tbin, cluster):
        {out_dir}/triplets_{suffix}_tbin{tbin}_cid{cluster}.png
    """
    os.makedirs(out_dir, exist_ok=True)

    df = load_transform_suffix_all_radars(suffix)
    if df.empty:
        print(f"[triplet-cands] suffix={suffix}: no detections.")
        return

    df = bin_time(df, BIN_SECONDS)
    # Radar colors
    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    n_plots = 0

    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue

        cents = compute_centroids(dc)
        if not cents:
            continue

        for cid, d_cluster in dc.groupby("cluster"):
            if cid == -1:
                continue  # skip noise

            centroid = cents.get(cid, None)
            if centroid is None:
                continue

            # --- build all candidate triplets for this cluster ---
            cands = build_candidates_for_cluster(
                d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR
            )
            if not cands:
                continue

            # --- compute dmid for each candidate & pick best ---
            dmid_vals = [
                triplet_features_for_scoring(c, centroid)[4]  # index 4 = dmid
                for c in cands
            ]
            best_idx = int(np.argmin(dmid_vals))

            # --- create figure for this (tbin, cid) ---
            plt.figure(figsize=(7, 7))

            # very light background: all cluster points
            plt.scatter(
                d_cluster["xw"],
                d_cluster["yw"],
                s=15,
                alpha=0.15,
                color="gray",
                label="cluster points",
            )

            # mark cluster centroid
            plt.scatter(
                centroid[0],
                centroid[1],
                s=120,
                marker="x",
                color="black",
                linewidths=1.5,
                label="cluster centroid",
            )

            # --- plot all candidate triplets ---
            selected_label_used = False
            candidate_label_used = False

            for i, cand in enumerate(cands):
                P = np.asarray(cand["points"])  # shape (3, 2)

                # connect triplet with line
                if i == best_idx:
                    # selected triplet – thicker, darker
                    plt.plot(
                        P[:, 0],
                        P[:, 1],
                        "-",
                        color="black",
                        linewidth=1.8,
                        alpha=0.9,
                        label="selected triplet" if not selected_label_used else None,
                    )
                    selected_label_used = True
                else:
                    # other candidates – light gray
                    plt.plot(
                        P[:, 0],
                        P[:, 1],
                        "-",
                        color="lightgray",
                        linewidth=0.6,
                        alpha=0.6,
                        label=(
                            "candidate triplets"
                            if not candidate_label_used
                            else None
                        ),
                    )
                    candidate_label_used = True

                # plot the 3 radar points in the triplet
                for j in range(3):
                    r = FUSION_RADARS[j]
                    col = radar_colors.get(r, "C0")

                    if i == best_idx:
                        s = 70
                        alpha = 1.0
                        ec = "black"
                        lw = 0.6
                        z = 4
                    else:
                        s = 35
                        alpha = 0.6
                        ec = "none"
                        lw = 0.0
                        z = 3

                    plt.scatter(
                        P[j, 0],
                        P[j, 1],
                        s=s,
                        alpha=alpha,
                        color=col,
                        edgecolor=ec,
                        linewidths=lw,
                        zorder=z,
                    )

            # --- radar anchors ---
            for r in FUSION_RADARS:
                if r not in radar_positions:
                    continue
                ax, ay = radar_positions[r]
                plt.scatter(
                    ax,
                    ay,
                    s=200,
                    marker="^",
                    color=radar_colors.get(r, "C0"),
                    edgecolor="black",
                    linewidths=1.2,
                    label=f"{radar_name_map[r]} anchor",
                )

            # cosmetics
            plt.axis("equal")
            plt.xlabel("x (m)", fontsize=34)
            plt.ylabel("y (m)", fontsize=34)
            plt.grid(True, linestyle="--", alpha=0.35)
           # plt.title(
            #    f"{suffix} — tbin={tbin}, cluster={cid}\n"
             #   f"All candidate triplets + selected (min-dmid)",
              #  fontsize=14,
               # weight="bold",
            #)
            plt.legend(loc="best", fontsize=16)

            out_path = os.path.join(
                out_dir, f"triplets_{suffix}_tbin{tbin}_cid{cid}.png"
            )
            plt.tight_layout()
            plt.tick_params(axis='both', which='major', labelsize=28)
            plt.tick_params(axis='both', which='minor', labelsize=26)
            plt.savefig(out_path, dpi=500, bbox_inches="tight")
            if show:
                plt.show()
            else:
                plt.close()

            print(f"[triplet-cands] saved {out_path}")
            n_plots += 1

            if max_plots is not None and n_plots >= max_plots:
                print(f"[triplet-cands] reached max_plots={max_plots}, stopping.")
                return
def plot_all_triplets_single_plot(
    suffix: str,
    out_dir: str = "DBSCAN_triplets_single",
    show: bool = False
):
    """
    Creates ONE single figure containing:
        - All DBSCAN centroids (black x)
        - All candidate triplets after time/spread gates (light gray)
        - Final selected triplets (thick black)
        - Triplet points (colored by radar)
        - Radar anchors

    This visualizes the entire suffix in one global plot.
    """
    os.makedirs(out_dir, exist_ok=True)

    df = load_transform_suffix_all_radars(suffix)
    if df.empty:
        print(f"[single-plot] suffix={suffix}: no detections.")
        return

    df = bin_time(df, BIN_SECONDS)
    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    # Collect all global items
    all_candidates = []    # each = Nx2 array of triplet points
    all_selected = []      # only the best candidate per cluster
    all_centroids = []     # (x,y)
    all_triplet_points = {r: [] for r in FUSION_RADARS}

    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue

        cents = compute_centroids(dc)
        if not cents:
            continue

        for cid, d_cluster in dc.groupby("cluster"):
            if cid == -1:
                continue

            centroid = cents[cid]
            all_centroids.append(centroid)

            # Build triplet candidates
            cands = build_candidates_for_cluster(
                d_cluster,
                centroid,
                FUSION_RADARS,
                K_NEAREST_PER_RADAR
            )
            if not cands:
                continue

            # Compute dmid & pick best
            dmid_vals = [
                triplet_features_for_scoring(c, centroid)[4] for c in cands
            ]
            best_idx = int(np.argmin(dmid_vals))

            # Save all candidates (except best labeled separately)
            for i, cand in enumerate(cands):
                P = np.asarray(cand["points"])  # (3,2)
                if i == best_idx:
                    all_selected.append(P)
                    # store radar-specific points
                    for j, r in enumerate(FUSION_RADARS):
                        all_triplet_points[r].append(P[j])
                else:
                    all_candidates.append(P)

    # ----------------------------------------------------
    # --- PLOT EVERYTHING IN ONE PLOT --------------------
    # ----------------------------------------------------
    plt.figure(figsize=(10, 9))

    # 1) All B/G candidates (thin gray triangles)
# 1) All candidate triplets (thin gray lines)
    candidate_label_used = False
    for P in all_candidates:
        plt.plot(
            P[:, 0], P[:, 1], "-",
            color="red",
            linewidth=0.6,
            alpha=0.6,
            label="Candidate triplets (after time + spread gate)" if not candidate_label_used else None
        )
        candidate_label_used = True
    

    # 2) Selected triplets (thick bold black)
# 2) Selected triplets (thick bold black)
    selected_label_used = False
    for P in all_selected:
        plt.plot(
            P[:, 0], P[:, 1], "-",
            color="black",
            linewidth=2.0,
            alpha=1.0,
            label="Selected triplets (min-dmid)" if not selected_label_used else None
        )
        selected_label_used = True

    # 3) Selected triplet points (colored by radar)
    for r in FUSION_RADARS:
        pts = np.array(all_triplet_points[r])
        if len(pts) > 0:
            plt.scatter(
                pts[:, 0],
                pts[:, 1],
                s=70,
                color=radar_colors[r],
                edgecolor="black",
                linewidths=0.7,
                label=f"{radar_name_map[r]} selected points"
            )

    # 4) DBSCAN centroids
    if all_centroids:
        cx, cy = zip(*all_centroids)
        plt.scatter(
            cx, cy,
            s=50,
            color="black",
            marker="X",
            linewidths=0.5,
            label="DBSCAN centroids",
    )

    # 5) Radar anchors
    for r in FUSION_RADARS:
        ax, ay = radar_positions[r]
        plt.scatter(
            ax, ay,
            s=260,
            marker="^",
            color=radar_colors[r],
            edgecolor="black",
            linewidths=1.4,
            label=f"{radar_name_map[r]} anchor"
        )

    # Final cosmetics
    plt.axis("equal")
    plt.xlabel("x (m)", fontsize=34)
    plt.ylabel("y (m)", fontsize=34)
    plt.grid(True, linestyle="--", alpha=0.35)
   # plt.title(f"{suffix} — All candidate triplets + selected (min-dmid)",
    #          fontsize=16, weight="bold")
    plt.legend(loc="best", fontsize=11)

    out_path = os.path.join(out_dir, f"{suffix}_all_triplets_single_plot.png")
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=28)
    plt.tick_params(axis='both', which='minor', labelsize=26)
    plt.savefig(out_path, dpi=500, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    print(f"[single-plot] saved {out_path}")

# Examples
plot_all_triplets_single_plot("c6")
# or
plot_all_triplets_single_plot("rr5")
plot_all_triplets_single_plot("r5")
plot_all_triplets_single_plot("cc2")



[single-plot] saved DBSCAN_triplets_single/c6_all_triplets_single_plot.png
[single-plot] saved DBSCAN_triplets_single/rr5_all_triplets_single_plot.png
[single-plot] saved DBSCAN_triplets_single/r5_all_triplets_single_plot.png
[single-plot] saved DBSCAN_triplets_single/cc2_all_triplets_single_plot.png


In [ ]:
def plot_final_triplets_and_targets(
    suffix: str,
    out_dir: str = "GBR_triplets_targets",
    show: bool = False
):
    """
    Plots ONLY:
        - final selected triplets (three radar points)
        - regression targets (x_avg, y_avg)

    Radar names shown as:
        rpi4 -> Radar 1
        rpi1 -> Radar 2
        rpi3 -> Radar 3
    """
    os.makedirs(out_dir, exist_ok=True)

    # 1) Get final selected triplets
    trips = select_triplets_for_suffix(suffix)
    if trips.empty:
        print(f"[final-triplets] suffix={suffix}: no selected triplets.")
        return

    plt.figure(figsize=(9, 8))

    # Map rpi names to friendly radar names
    radar_name_map = {
        "rpi4": "Radar 1",
        "rpi1": "Radar 2",
        "rpi3": "Radar 3",
    }

    # Radar colors
    radar_colors = {
        "rpi4": "#6baced",  # Radar 1
        "rpi1": "#d98750",  # Radar 2
        "rpi3": "#cde06c",  # Radar 3
    }

    # 2) Plot radar triplet points
    point_label_used = {r: False for r in FUSION_RADARS}

    for _, row in trips.iterrows():
        for r in FUSION_RADARS:
            xw = row[f"{r}_xw"]
            yw = row[f"{r}_yw"]

            plt.scatter(
                xw, yw,
                s=70,
                color=radar_colors[r],
                edgecolor="black",
                linewidths=0.7,
                alpha=1.0,
                label=f"{radar_name_map[r]} triplet point"
                      if not point_label_used[r] else None,
            )
            point_label_used[r] = True

    # 3) Plot regression targets
    plt.scatter(
        trips["x_avg"], trips["y_avg"],
        s=90,
        marker="+",
        color="#f50511",
        edgecolor="black",
        linewidths=4,
        alpha=1.0,
        label="Regression target (x_avg, y_avg)"
    )

    # 4) Plot radar anchors
    for r in FUSION_RADARS:
        ax, ay = radar_positions[r]
        plt.scatter(
            ax, ay,
            s=260,
            marker="^",
            color=radar_colors[r],
            edgecolor="black",
            linewidths=1.3,
            label=f"{radar_name_map[r]} anchor"
        )

    # Cosmetics
    plt.axis("equal")
    plt.xlabel("x (m)", fontsize=34)
    plt.ylabel("y (m)", fontsize=34)
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(loc="best", fontsize=16)

    # Save
    out_path = os.path.join(out_dir, f"{suffix}_final_triplets_targets.png")
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=28)
    plt.tick_params(axis='both', which='minor', labelsize=26)
    plt.savefig(out_path, dpi=500, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    print(f"[final-triplets] saved {out_path}")
plot_final_triplets_and_targets("rr5")
plot_final_triplets_and_targets("c6")
plot_final_triplets_and_targets("cc2")
plot_final_triplets_and_targets("r5")